# Electrochemical Analysis Suite - Jupyter Example

This notebook demonstrates how to use the clean implementation from Jupyter for electrochemical data analysis.

## Features Demonstrated
- Cell management
- File processing
- Data visualization
- ActionID management

In [ ]:
# Setup - Add project root to path
import sys
from pathlib import Path

# Add project root to path
project_root = Path().absolute()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the backend API
from src_clean.backend import get_backend_api

# Initialize backend
api = get_backend_api()
print("✅ Backend API initialized")
print(f"Database: data_clean/electrochemical.db")

## 1. Database Statistics

In [ ]:
# Get database statistics
stats = api.get_database_stats()

print("📊 Database Statistics:")
print(f"• Cells: {stats['cell_count']}")
print(f"• Files: {stats['file_count']}")
print(f"• Segments: {stats['segment_count']}")
print(f"• User ActionID Mappings: {stats['user_mappings_count']}")
print(f"• Database Size: {stats['database_size_mb']:.2f} MB")
print(f"• Supported Instruments: {', '.join(stats['supported_instruments'])}")

## 2. Cell Management

In [ ]:
# Create a new cell
result = api.create_cell(
    name="JUPYTER_CELL_001",
    chemistry="Li_ion",
    notes="Example cell created from Jupyter notebook"
)

if result.success:
    print(f"✅ {result.message}")
else:
    print(f"❌ {result.error}")

In [ ]:
# List all cells
cells = api.get_cells()

print(f"📋 Found {len(cells)} cells:")
for cell in cells:
    print(f"• {cell['name']} ({cell['chemistry']}) - {cell['file_count']} files")
    if cell['notes']:
        print(f"  Notes: {cell['notes']}")

## 3. ActionID Mappings

In [ ]:
# List ActionID mappings
import pandas as pd

mappings = api.get_actionid_mappings()
df_mappings = pd.DataFrame(mappings)

print("🔧 ActionID Mappings:")
display(df_mappings[['action_id', 'technique_name', 'fundamental_technique', 'user_defined']])

In [ ]:
# Add a custom ActionID mapping
result = api.add_actionid_mapping(
    action_id=99,
    technique_name="Custom Technique",
    fundamental_technique="custom"
)

if result.success:
    print(f"✅ {result.message}")
else:
    print(f"❌ {result.error}")

## 4. File Processing (if test files available)

In [ ]:
# Check for test files
from pathlib import Path

test_par = Path("data/measurement_groups/GITT_EIS_Charge_cycle1_Channel 2.par")
test_csv = Path("data/measurement_groups/GITT_EIS_Charge_cycle1_Channel 2.par.csv")

if test_par.exists() and test_csv.exists():
    print(f"📁 Found test files:")
    print(f"• PAR: {test_par.name} ({test_par.stat().st_size:,} bytes)")
    print(f"• CSV: {test_csv.name} ({test_csv.stat().st_size:,} bytes)")
    
    # Validate files
    validation = api.validate_dual_files(test_par, test_csv)
    print(f"• Validation: {'✅ Pass' if validation['success'] else '❌ Fail'}")
    
    # Process files
    if validation['success']:
        print("\n🔄 Processing files...")
        result = api.process_dual_files(test_par, test_csv, "JUPYTER_CELL_001")
        
        if result.success:
            print(f"✅ {result.message}")
            print(f"📄 File ID: {result.file_id}")
            
            # Store file_id for data analysis
            processed_file_id = result.file_id
        else:
            print(f"❌ {result.error}")
else:
    print("📁 Test files not found - skipping file processing demo")
    print("   Place .par/.par.csv files in data/measurement_groups/ to test processing")

## 5. Data Analysis (if files were processed)

In [ ]:
# Analyze processed data (if available)
try:
    if 'processed_file_id' in locals():
        # Get file data
        data = api.get_file_data(processed_file_id)
        
        if data is not None:
            print(f"📊 Data Analysis for {processed_file_id}:")
            print(f"• Total rows: {data.height:,}")
            print(f"• Columns: {data.width}")
            print(f"• Time range: {data['time_s'].min():.1f} - {data['time_s'].max():.1f} seconds")
            
            # Convert to pandas for analysis
            df = data.to_pandas()
            
            # Show first few rows
            print("\n📋 First 5 rows:")
            display(df.head())
            
            # Show data types
            print("\n🔍 Column types:")
            for col in df.columns:
                print(f"• {col}: {df[col].dtype}")
        else:
            print("❌ Could not load processed data")
    else:
        print("ℹ️ No processed data available for analysis")
except Exception as e:
    print(f"❌ Error in data analysis: {e}")

## 6. Basic Visualization (if data available)

In [ ]:
# Create basic plots if data is available
import matplotlib.pyplot as plt

try:
    if 'df' in locals() and not df.empty:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        fig.suptitle(f'Electrochemical Data Analysis - {processed_file_id}')
        
        # Plot 1: Potential vs Time
        if 'potential_v' in df.columns and 'time_s' in df.columns:
            axes[0,0].plot(df['time_s'], df['potential_v'], 'b-', linewidth=0.8)
            axes[0,0].set_xlabel('Time (s)')
            axes[0,0].set_ylabel('Potential (V)')
            axes[0,0].set_title('Potential vs Time')
            axes[0,0].grid(True, alpha=0.3)
        
        # Plot 2: Current vs Time
        if 'current_a' in df.columns and 'time_s' in df.columns:
            axes[0,1].plot(df['time_s'], df['current_a'], 'r-', linewidth=0.8)
            axes[0,1].set_xlabel('Time (s)')
            axes[0,1].set_ylabel('Current (A)')
            axes[0,1].set_title('Current vs Time')
            axes[0,1].grid(True, alpha=0.3)
        
        # Plot 3: I-V Curve
        if 'potential_v' in df.columns and 'current_a' in df.columns:
            axes[1,0].plot(df['current_a'], df['potential_v'], 'g-', linewidth=0.8)
            axes[1,0].set_xlabel('Current (A)')
            axes[1,0].set_ylabel('Potential (V)')
            axes[1,0].set_title('I-V Curve')
            axes[1,0].grid(True, alpha=0.3)
        
        # Plot 4: Power vs Time
        if 'power_w' in df.columns and 'time_s' in df.columns:
            axes[1,1].plot(df['time_s'], df['power_w'], 'm-', linewidth=0.8)
            axes[1,1].set_xlabel('Time (s)')
            axes[1,1].set_ylabel('Power (W)')
            axes[1,1].set_title('Power vs Time')
            axes[1,1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("📈 Plots generated successfully!")
    else:
        print("ℹ️ No data available for visualization")
except Exception as e:
    print(f"❌ Error in visualization: {e}")

## 7. Summary

In [ ]:
# Final summary
final_stats = api.get_database_stats()

print("🏁 Session Summary:")
print(f"• Total cells: {final_stats['cell_count']}")
print(f"• Total files: {final_stats['file_count']}")
print(f"• Total segments: {final_stats['segment_count']}")
print(f"• Database size: {final_stats['database_size_mb']:.2f} MB")

print("\n✅ Jupyter notebook example completed!")
print("\n🔧 Next steps:")
print("• Use the CLI: python echem_cli.py --help")
print("• Use the Qt GUI: python echem_gui.py")
print("• Process your own .par/.par.csv files")
print("• Explore the universal schema data")